In [1]:
# Установка необходимых библиотек.

# !pip install torch-geometric
# !pip install sentence_transformers

In [2]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.datasets import MovieLens
from torch_geometric.nn import GCNConv, SAGEConv, to_hetero
import torch_geometric.transforms as T
import pandas as pd
import numpy as np

/Users/slavyan/Desktop/cnn_lectures/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset_path = '/tmp/'
dataset = MovieLens(root=dataset_path)

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
data = dataset[0].to(device)

In [ ]:
### Ноды пользователей
data['user']

{'num_nodes': 610}

In [9]:
# Add user node features for message passing:
data['user'].x = torch.eye(data['user'].num_nodes, device=device)
del data['user'].num_nodes

In [14]:
data['user']['x']

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]])

In [10]:
data['rates']

{'edge_index': tensor([[   0,    0,    0,  ...,  609,  609,  609],
        [   0,    2,    5,  ..., 9462, 9463, 9503]]), 'edge_label': tensor([4, 4, 4,  ..., 5, 5, 3]), 'time': tensor([ 964982703,  964981247,  964982224,  ..., 1494273047, 1493846352,
        1493846415])}

In [15]:
torch.bincount(data['user', 'movie'].edge_label)


tensor([ 1370,  4602, 13101, 33183, 35369, 13211])

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = dataset[0].to(device)

# Add user node features for message passing:
data['user'].x = torch.eye(data['user'].num_nodes, device=device)
del data['user'].num_nodes

# Add a reverse ('movie', 'rev_rates', 'user') relation for message passing:
data = T.ToUndirected()(data)
del data['movie', 'rev_rates', 'user'].edge_label  # Remove "reverse" label.

# Perform a link-level split into training, validation, and test edges:
train_data, val_data, test_data = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    neg_sampling_ratio=0.0,
    edge_types=[('user', 'rates', 'movie')],
    rev_edge_types=[('movie', 'rev_rates', 'user')],
)(data)

In [17]:
weight = torch.bincount(train_data['user', 'movie'].edge_label)
weight = weight.max() / weight


In [14]:

def weighted_mse_loss(pred, target, weight=None):
    weight = 1. if weight is None else weight[target].to(pred.dtype)
    return (weight * (pred - target.to(pred.dtype)).pow(2)).mean()


class GNNEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x


class EdgeDecoder(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.lin1 = Linear(2 * hidden_channels, hidden_channels)
        self.lin2 = Linear(hidden_channels, 1)

    def forward(self, z_dict, edge_label_index):
        row, col = edge_label_index
        z = torch.cat([z_dict['user'][row], z_dict['movie'][col]], dim=-1)

        z = self.lin1(z).relu()
        z = self.lin2(z)
        return z.view(-1)


class Model(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.encoder = GNNEncoder(hidden_channels, hidden_channels)
        self.encoder = to_hetero(self.encoder, data.metadata(), aggr='sum')
        self.decoder = EdgeDecoder(hidden_channels)

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        z_dict = self.encoder(x_dict, edge_index_dict)
        return self.decoder(z_dict, edge_label_index)


model = Model(hidden_channels=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [16]:
def train():
    model.train()
    optimizer.zero_grad()
    pred = model(train_data.x_dict, train_data.edge_index_dict,
                 train_data['user', 'movie'].edge_label_index)
    target = train_data['user', 'movie'].edge_label
    loss = weighted_mse_loss(pred, target, weight)
    loss.backward()
    optimizer.step()
    return float(loss)


@torch.no_grad()
def test(data):
    model.eval()
    pred = model(data.x_dict, data.edge_index_dict,
                 data['user', 'movie'].edge_label_index)
    pred = pred.clamp(min=0, max=5)
    target = data['user', 'movie'].edge_label.float()
    rmse = F.mse_loss(pred, target).sqrt()
    return float(rmse)


for epoch in range(1, 301):
    loss = train()
    train_rmse = test(train_data)
    val_rmse = test(val_data)
    test_rmse = test(test_data)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train: {train_rmse:.4f}, '
          f'Val: {val_rmse:.4f}, Test: {test_rmse:.4f}')

Epoch: 001, Loss: 19.3041, Train: 3.4078, Val: 3.4192, Test: 3.4047
Epoch: 002, Loss: 18.0009, Train: 3.1493, Val: 3.1622, Test: 3.1482
Epoch: 003, Loss: 15.4180, Train: 2.6743, Val: 2.6897, Test: 2.6768
Epoch: 004, Loss: 11.4339, Train: 1.8990, Val: 1.9163, Test: 1.9061
Epoch: 005, Loss: 7.1346, Train: 1.0947, Val: 1.0902, Test: 1.0925
Epoch: 006, Loss: 7.1619, Train: 1.1725, Val: 1.1494, Test: 1.1579
Epoch: 007, Loss: 9.4187, Train: 1.0820, Val: 1.0762, Test: 1.0786
Epoch: 008, Loss: 7.1914, Train: 1.3225, Val: 1.3338, Test: 1.3276
Epoch: 009, Loss: 5.9393, Train: 1.6931, Val: 1.7083, Test: 1.6983
Epoch: 010, Loss: 6.3964, Train: 1.9274, Val: 1.9428, Test: 1.9314
Epoch: 011, Loss: 7.1489, Train: 1.9962, Val: 2.0115, Test: 1.9996
Epoch: 012, Loss: 7.4153, Train: 1.9309, Val: 1.9464, Test: 1.9345
Epoch: 013, Loss: 7.1284, Train: 1.7622, Val: 1.7778, Test: 1.7665
Epoch: 014, Loss: 6.5063, Train: 1.5243, Val: 1.5393, Test: 1.5292
Epoch: 015, Loss: 5.8941, Train: 1.2783, Val: 1.2904, Test

### Задание

Подберите оптимальные параметры для сети из примера выше(2 балла)

Попробуйте вместо GraphSage модуль Graph Attention и также подберите оптимальные параметры (2 балла)